In [ ]:
# ==============================================================================
# PROYECTO ENTERPRISE BRAIN - GRUPO G8
# Semana 8: Flujo Documento (OpenAlex API) -> Grafo Neo4j
# Responsable: Alexander Chamorro
# ==============================================================================

# 1. INSTALACIONES Y DEPENDENCIAS (Ejecutar en la primera celda)
import os
print("🔄 Instalando dependencias necesarias (Tesseract OCR y librerías de Python)...")
os.system("apt-get update -qq")
os.system("apt-get install -y tesseract-ocr tesseract-ocr-spa poppler-utils")
os.system("pip install -q PyMuPDF pandas pytesseract pdf2image neo4j requests")
print("✅ Instalaciones completadas.\n")

import fitz
import pytesseract
from PIL import Image
import io
import pandas as pd
import re
import unicodedata
from collections import defaultdict
from neo4j import GraphDatabase
import requests

🔄 Instalando dependencias necesarias (Tesseract OCR y librerías de Python)...
✅ Instalaciones completadas.



In [ ]:
# ==============================================================================
# 2. CONFIGURACIÓN Y REGLAS DE EXTRACCIÓN (Alineadas al .owl)
# ==============================================================================
CORDILLERAS = ["Blanca", "Negra", "Huallanca", "Huayhuash", "Raura"]
DEPARTAMENTOS = ["Áncash", "Ancash", "Junín", "Cusco", "Lima", "Huánuco", "La Libertad"]
POBLADOS_CONOCIDOS = ["Huaraz", "Carhuaz", "Yungay", "Caraz", "Huallanca", "Conchucos"]
INSTITUCIONES = {
    "INAIGEM": r"INAIGEM|Instituto Nacional de Investigaci[oó]n en Glaciares",
    "ANA": r"\bANA\b|Autoridad Nacional del Agua",
    "Corporación Peruana del Santa": r"Corporaci[oó]n Peruana del Santa|C\.P\.S\.",
    "Electroperú": r"Electroper[uú]|ELECTROPER[UÚ]"
}

def norm(t):
    return re.sub(r"\s+", " ", "".join(c for c in unicodedata.normalize("NFD", t or "") if unicodedata.category(c) != "Mn").lower()).strip()

def _re(p, flags=0): return re.compile(p, flags)

LETRA = "A-ZÁÉÍÓÚÑÜ"
PAL = rf"[{LETRA}][A-Za-zÁÉÍÓÚÑÜáéíóúñü'\-]+"
LISTA = rf"(?:{PAL}|\d{{2,3}}(?!\d))(?:\s+(?:Alta|Baja|Grande|Chico|Norte|Sur|Este|Oeste|\d+))?(?:\s*(?:,|\by\b|\be\b)\s*(?:(?:la|el)\s+)?(?:{PAL}|\d{{2,3}}))*"

REGLAS_ENTIDADES = [
    ("Laguna", _re(rf"\b(?:[Ll]agunas?|LAGUNAS?)\s+(?:(?:de|del)\s+)?(?P<n>{LISTA})")),
    ("Glaciar", _re(rf"\b(?:[Gg]laciar(?:es)?|GLACIAR(?:ES)?)\s+(?:(?:de|del)\s+)?(?P<n>{LISTA})")),
    ("Nevado", _re(rf"\b(?:[Nn]evados?|NEVADOS?)\s+(?:(?:de|del)\s+)?(?P<n>{LISTA})")),
    ("Cordillera", _re(r"\b[Cc]ordillera(?:s)?\s+(?P<n>" + "|".join(sorted(CORDILLERAS, key=len, reverse=True)) + r")\b")),
    ("Departamento", _re(r"\b(?:[Dd]epartamento|[Rr]egi[oó]n)\s+(?:de\s+)?(?P<n>" + "|".join(sorted(DEPARTAMENTOS, key=len, reverse=True)) + r")\b")),
    ("CentroPoblado", _re(r"\b(?P<n>" + "|".join(sorted(POBLADOS_CONOCIDOS, key=len, reverse=True)) + r")\b")),
]

def menciones_reglas(texto):
    out = []
    for tipo, pat in REGLAS_ENTIDADES:
        for m in pat.finditer(texto):
            for n in re.split(r"\s*(?:,|\by\b|\be\b)\s*", m.group("n")):
                n_limpio = re.sub(r"^(la|el|los|las)\s+", "", n, flags=re.I).strip(" ,.;:()-")
                if len(n_limpio) > 2:
                    out.append({"tipo": tipo, "nombre": n_limpio.capitalize(), "ini": m.start("n"), "metodo": "reglas"})
    for nombre, p in INSTITUCIONES.items():
        for m in re.finditer(p, texto):
            out.append({"tipo": "Institucion", "nombre": nombre, "ini": m.start(), "metodo": "reglas"})
    return out

def extraer_chunk(c):
    texto = c["text"]
    pagina = c.get("pagina", 1)
    menc = menciones_reglas(texto)
    trip = []
    base = {"paper_id": c.get("paper_id", "nuevo_doc"), "page_start": pagina, "evidence": texto.strip()}

    por_tipo = defaultdict(list)
    for m in menc: por_tipo[m["tipo"]].append(m)

    if re.search(r"ubicad|situad|se encuentra|en el departamento", texto.lower()):
        for s in por_tipo["Laguna"] + por_tipo["Glaciar"] + por_tipo["CentroPoblado"]:
            for ob in por_tipo["Cordillera"] + por_tipo["Departamento"]:
                trip.append({**base, "subject": s["nombre"], "subject_type": s["tipo"], "predicate": "ubicadaEn", "object": ob["nombre"], "object_type": ob["tipo"]})

    for m in re.finditer(r"(?:riesgo|peligro)\s+(?:(?:es|de nivel)\s+)?(?P<v>muy\s+alto|alto|medio|bajo)\b", texto, re.I):
        v = re.sub(r"\s+", " ", m.group("v").lower()).capitalize()
        if por_tipo["Laguna"]:
            trip.append({**base, "subject": por_tipo["Laguna"][0]["nombre"], "subject_type": "Laguna", "predicate": "tieneNivelRiesgo", "object": v, "object_type": "NivelRiesgo"})

    if por_tipo["Institucion"] and (por_tipo["Laguna"] or por_tipo["Glaciar"]):
        for s in por_tipo["Institucion"]:
            for ob in por_tipo["Laguna"] + por_tipo["Glaciar"]:
                trip.append({**base, "subject": s["nombre"], "subject_type": "Institucion", "predicate": "intervieneEn", "object": ob["nombre"], "object_type": ob["tipo"]})

    if re.search(r"amenaza|afecta|aguas abajo|peligro para", texto.lower()):
        for s in por_tipo["Laguna"]:
            for ob in por_tipo["CentroPoblado"]:
                trip.append({**base, "subject": s["nombre"], "subject_type": "Laguna", "predicate": "amenazaA", "object": ob["nombre"], "object_type": "CentroPoblado"})

    if re.search(r"contacto con|alimentad|deshielo|pie del nevado", texto.lower()):
        for s in por_tipo["Laguna"]:
            for ob in por_tipo["Glaciar"] + por_tipo["Nevado"]:
                trip.append({**base, "subject": s["nombre"], "subject_type": "Laguna", "predicate": "alimentadaPor", "object": ob["nombre"], "object_type": ob["tipo"]})

    return trip

In [ ]:
# ==============================================================================
# 3. MÓDULO DE INTEGRACIÓN CON SEMANTIC_SCHOLAR
# ==============================================================================
import requests
import re

def limpiar_nombre_archivo(titulo):
    # Reemplaza espacios y caracteres especiales por guiones bajos para que sea seguro en Linux/Colab
    nombre_limpio = re.sub(r'[^a-zA-Z0-9]', '_', titulo)
    # Limitamos a 60 caracteres para evitar nombres de archivo excesivamente largos
    return f"/content/{nombre_limpio[:60]}.pdf"

def descargar_pdf_desde_openalex(query_busqueda="laguna Palcacocha riesgo Cordillera Blanca"):
    print(f"\n🔍 Consultando la API para: '{query_busqueda}'...")
    url = "https://api.openalex.org/works"
    params = {"search": query_busqueda, "per-page": 15}

    try:
        response = requests.get(url, params=params)
        results = response.json().get("results", [])

        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
            'Accept': 'application/pdf'
        }

        palabras_clave = ["glaciar", "glacier", "laguna", "lake", "cordillera", "palcacocha", "paron", "riesgo", "hazard", "andes"]

        for work in results:
            titulo = work.get('title', 'Sin_titulo')

            # Filtro de relevancia
            if not any(palabra in titulo.lower() for palabra in palabras_clave):
                continue

            oa = work.get("open_access", {})
            url_candidata = oa.get("oa_url")

            if url_candidata and ("pdf" in url_candidata.lower()):
                # AQUÍ SE GENERA LA RUTA DINÁMICA
                ruta_dinamica = limpiar_nombre_archivo(titulo)

                print(f"\n📚 Evaluando documento: {titulo}")
                print(f"📥 Descargando en: {ruta_dinamica}")

                try:
                    pdf_resp = requests.get(url_candidata, headers=headers, timeout=15)
                    if b"%PDF" in pdf_resp.content[:10]:
                        with open(ruta_dinamica, "wb") as f:
                            f.write(pdf_resp.content)
                        print(f"✅ ¡Éxito! Archivo guardado de forma única.")
                        return ruta_dinamica # Devuelve el nombre específico de este paper
                except Exception as e:
                    print(f"⚠️ Error al descargar: {e}")
        return None
    except Exception as e:
        print(f"❌ Error de API: {e}")
        return None

In [ ]:
# ==============================================================================
# 4. EXTRACCIÓN DE TEXTO (Lectura de PDF + OCR Robusto por página)
# ==============================================================================
def procesar_nuevo_pdf(ruta_pdf):
    print(f"\n📄 Leyendo el documento: {ruta_pdf}")
    texto_completo = ""
    chunks = []
    try:
        pdf_doc = fitz.open(ruta_pdf)
        print(f"✅ Archivo abierto correctamente. Total de páginas: {len(pdf_doc)}")

        texto_nativo = ""
        for page in pdf_doc:
            texto_nativo += page.get_text()

        if len(texto_nativo.strip()) < 100:
            print("👁️ PDF escaneado detectado. Aplicando OCR seguro página por página...")
            paginas_a_procesar = min(15, len(pdf_doc))
            print(f"⚠️ Analizando las primeras {paginas_a_procesar} páginas...")

            for i in range(paginas_a_procesar):
                print(f"   Escaneando página {i+1}...")
                page = pdf_doc[i]
                pix = page.get_pixmap(dpi=150)
                img_bytes = pix.tobytes("png")

                imagen = Image.open(io.BytesIO(img_bytes))
                texto_pagina = pytesseract.image_to_string(imagen, lang='spa')

                parrafos = texto_pagina.split("\n\n")
                for p in parrafos:
                    if len(p.strip()) > 30:
                        chunks.append({"text": p.strip(), "paper_id": ruta_pdf.split('/')[-1], "pagina": i+1})
        else:
            print("✅ Texto nativo encontrado.")
            for i, page in enumerate(pdf_doc):
                t_pag = page.get_text()
                parrafos = t_pag.split("\n\n")
                for p in parrafos:
                    if len(p.strip()) > 30:
                        chunks.append({"text": p.strip(), "paper_id": ruta_pdf.split('/')[-1], "pagina": i+1})

        todas_tripletas = []
        for c in chunks:
            todas_tripletas.extend(extraer_chunk(c))

        print(f"✅ Extracción finalizada. Se encontraron {len(todas_tripletas)} relaciones.")
        return todas_tripletas

    except Exception as e:
        print(f"❌ Error al procesar el PDF: {e}")
        return []

In [ ]:
# ==============================================================================
# 5. CARGA A LA BASE DE DATOS NEO4J
# ==============================================================================
REL_NEO4J = {
    "ubicadaEn": "UBICADA_EN",
    "tieneNivelRiesgo": "TIENE_NIVEL_RIESGO",
    "intervieneEn": "INTERVIENE_EN",
    "amenazaA": "AMENAZA_A",
    "alimentadaPor": "ALIMENTADA_POR"
}

def subir_a_neo4j(tripletas, uri, user, password):
    if not tripletas:
        print("⚠️ No hay datos para cargar en Neo4j.")
        return

    print("\n☁️ Conectando a Neo4j...")
    try:
        driver = GraphDatabase.driver(uri, auth=(user, password))
        with driver.session() as session:
            for t in tripletas:
                sid, stipo, s = f"{t['subject_type']}:{norm(t['subject'])}", t["subject_type"], t["subject"]
                oid, otipo, o = f"{t['object_type']}:{norm(t['object'])}", t["object_type"], t["object"]
                rel_type = REL_NEO4J.get(t["predicate"], "RELACIONADO_CON")

                query = f"""
                MERGE (a:Entidad {{id: $sid}}) ON CREATE SET a.nombre = $s, a.tipo = $stipo SET a:{stipo}
                MERGE (b:Entidad {{id: $oid}}) ON CREATE SET b.nombre = $o, b.tipo = $otipo SET b:{otipo}
                MERGE (a)-[r:{rel_type}]->(b)
                ON CREATE SET r.documento = $doc, r.pagina = $pag, r.evidencia = $evidencia
                """
                session.run(query, sid=sid, s=s, stipo=stipo, oid=oid, o=o, otipo=otipo,
                            doc=t["paper_id"], pag=t["page_start"], evidencia=t["evidence"][:200])
        driver.close()
        print("🎉 ¡Nuevos datos de OpenAlex integrados exitosamente al Grafo!")
    except Exception as e:
        print(f"❌ Error Neo4j: {e}")

In [ ]:
# ==============================================================================
# 6. EJECUCIÓN EN LOTE DEL FLUJO COMPLETO (OpenAlex -> Extracción -> Neo4j)
# ==============================================================================
import os
import pandas as pd

# Recuerda colocar las credenciales exactas de tu instancia AuraDB
CREDENCIALES_NEO4J = {
    "URI": "neo4j+s://1c49885b.URI.neo4j.io",
    "USER": "YOUR_USER",
    "PASSWORD": "YOUR_PASSWORD"
}

# Lista de términos estratégicos (puedes agregar los que necesites para tu grupo)
terminos_de_busqueda = [
    "laguna Palcacocha riesgo Ancash",
    "glaciar Paron amenaza",
    "glacier lake outburst flood Huaraz",
    "INAIGEM glaciares Peru"
]

tripletas_totales = []

print("🚀 Iniciando el flujo de extracción en lote...")

for termino in terminos_de_busqueda:
    print(f"\n{'-'*70}")

    # 1. Buscar y descargar con nombre dinámico
    ruta_pdf = descargar_pdf_desde_openalex(query_busqueda=termino)

    if ruta_pdf:
        # Verificación opcional para no reprocesar si ya se analizó en la sesión actual de Colab
        if not os.path.exists(ruta_pdf):
             print(f"⚠️ El archivo {ruta_pdf} no se encuentra en el sistema de archivos de Colab.")
             continue

        # 2. Procesar el PDF y extraer entidades/relaciones
        tripletas_nuevas = procesar_nuevo_pdf(ruta_pdf)

        if tripletas_nuevas:
            tripletas_totales.extend(tripletas_nuevas)

            # 3. Inyección directa al Grafo
            #subir_a_neo4j(
            #    tripletas=tripletas_nuevas,
            #    uri=CREDENCIALES_NEO4J["URI"],
            #    user=CREDENCIALES_NEO4J["USER"],
            #    password=CREDENCIALES_NEO4J["PASSWORD"]
            #)
        else:
            print("⚠️ El documento se descargó, pero las reglas PLN no detectaron relaciones del dominio.")
    else:
        print("⏭️ Pasando al siguiente término de búsqueda...")

# 4. Tabla de control final
if tripletas_totales:
    print(f"\n{'-'*70}")
    print(f"📊 RESUMEN DE LA SESIÓN: Se inyectaron {len(tripletas_totales)} relaciones al grafo general.")
    df_final = pd.DataFrame(tripletas_totales)
    from IPython.display import display
    display(df_final[['subject', 'predicate', 'object', 'paper_id', 'page_start']].drop_duplicates().head(20))
else:
    print("\n❌ Finalizado. No se extrajo ninguna relación válida de los términos consultados.")


🚀 Iniciando el flujo de extracción en lote...

----------------------------------------------------------------------

🔍 Consultando la API para: 'laguna Palcacocha riesgo Ancash'...

📚 Evaluando documento: Reconstruction of the 1941 GLOF process chain at Lake Palcacocha (Cordillera Blanca, Peru)
📥 Descargando en: /content/Reconstruction_of_the_1941_GLOF_process_chain_at_Lake_Palcac.pdf
✅ ¡Éxito! Archivo guardado de forma única.

📄 Leyendo el documento: /content/Reconstruction_of_the_1941_GLOF_process_chain_at_Lake_Palcac.pdf
✅ Archivo abierto correctamente. Total de páginas: 22
✅ Texto nativo encontrado.
✅ Extracción finalizada. Se encontraron 9 relaciones.

☁️ Conectando a Neo4j...
🎉 ¡Nuevos datos de OpenAlex integrados exitosamente al Grafo!

----------------------------------------------------------------------

🔍 Consultando la API para: 'glaciar Paron amenaza'...
⏭️ Pasando al siguiente término de búsqueda...

----------------------------------------------------------------------

,subject,predicate,object,paper_id,page_start
0,ANA,intervieneEn,513,Reconstruction_of_the_1941_GLOF_process_chain_...,2
1,ANA,intervieneEn,Palcacocha,Reconstruction_of_the_1941_GLOF_process_chain_...,19
4,Electroperú,intervieneEn,Palcacocha,Reconstruction_of_the_1941_GLOF_process_chain_...,19
5,Electroperú,intervieneEn,Palcacocha,Reconstruction_of_the_1941_GLOF_process_chain_...,21
6,Electroperú,intervieneEn,Elec,Reconstruction_of_the_1941_GLOF_process_chain_...,21
7,Electroperú,intervieneEn,Huaráz,Reconstruction_of_the_1941_GLOF_process_chain_...,21
8,Electroperú,intervieneEn,Boletin,Reconstruction_of_the_1941_GLOF_process_chain_...,21
9,ANA,intervieneEn,Perú,Multi_Decadal_Glacier_Area_and_Mass_Balance_Ch...,12
10,INAIGEM,intervieneEn,Futuras,Multi_Decadal_Glacier_Area_and_Mass_Balance_Ch...,13
11,INAIGEM,intervieneEn,Perú,Multi_Decadal_Glacier_Area_and_Mass_Balance_Ch...,13


### Para poder visualizar el grafo, puedes crear una cuenta de Neo4j, ir al apartado de una instancia, crear una gratis y los datos que se generen, reemplazarlos en los valores de la sección 6